# Get DAILY Data for Basin

In [2]:
%load_ext autoreload
%autoreload 2

import logging

import geopandas as gpd
import pandas as pd
from mergedownloader.downloader import Downloader
from mergedownloader.file_downloader import ConnectionType, DownloadMode, FileDownloader
from mergedownloader.inpeparser import INPE_SERVER, InpeParsers, InpeTypes
from mergedownloader.utils import GISUtil

from utils.hydrology import Hydrology


## Constants

In [6]:
BASIN = "BarraDosBugres"
STATION = "66010000"

## Load Basin boundaries

In [1]:
basin = gpd.read_parquet(f"/data/CN3S/basins/{BASIN}/DA_Station_{STATION}.parquet")
basin.plot()


NameError: name 'gpd' is not defined

In [ ]:
# calculate the area in squared kilometers of basin (need to reproject to an equal-area projection first)
basin = basin.to_crs("epsg:32723")
basin["area_km2"] = basin.geometry.area / 1e6
print(f"Basin area: {basin['area_km2'].iloc[0]:.2f} km²")

Basin area: 9642.06 km²


## Create Downloader

In [ ]:
fd = FileDownloader(
    server=INPE_SERVER,
    connection_type=ConnectionType.HTTP,
    download_mode=DownloadMode.NO_UPDATE,
    log_level=logging.DEBUG,
)

downloader = Downloader(
    file_downloader=fd,
    local_folder="/data/CN3S/downloads",
    parsers=InpeParsers,
    log_level=logging.DEBUG,
)


Using wget through HTTP on: ftp.cptec.inpe.br


In [4]:
da = downloader.open_file("2026-05-01", InpeTypes.MONTHLY_ACCUM_YEARLY)
da

<xarray.DataArray 'pacum' (time: 1, latitude: 691, longitude: 551)> Size: 3MB
[380741 values with dtype=float64]
Coordinates:
  * longitude    (longitude) float64 4kB -85.05 -84.95 -84.85 ... -30.15 -30.05
  * latitude     (latitude) float64 6kB -56.15 -56.05 -55.95 ... 12.75 12.85
  * time         (time) datetime64[ns] 8B 2026-05-01T12:00:00
    spatial_ref  int64 8B 0

## Get DAILY MERGE Rain for Basin

Considering the size of the cube, we will split the rain in years then concatenate everything into one pandas dataframe. 

In [ ]:
years = range(2000, 2027)
rain = pd.DataFrame()

for year in years:
    print(f"Processing year {year}...")

    cube_year = downloader.create_cube(
        f"{year}-01-01",
        f"{year}-12-31",
        InpeTypes.DAILY_RAIN,
    )
    cube_year_clipped = GISUtil.cut_cube_by_geoms(cube_year, basin.geometry)

    rain_year = cube_year_clipped.mean(dim=["latitude", "longitude"]).to_dataframe()
    rain_year = rain_year.drop(columns=["spatial_ref", "valid_time", "surface"])
    rain_year = rain_year.rename(columns={"rdp": "prec"})

    rain_year.to_parquet(f"/data/CN3S/basins/{BASIN}/Daily_Prec_{year}_{STATION}.parquet")

    rain = pd.concat([rain, rain_year], axis=0)

rain.to_parquet(f"/data/CN3S/basins/{BASIN}/Daily_Prec_{STATION}.parquet")

Processing year 2000...
Processing year 2001...
Processing year 2002...
Processing year 2003...
Processing year 2004...
Processing year 2005...
Processing year 2006...
Processing year 2007...
Processing year 2008...
Processing year 2009...
Processing year 2010...
Processing year 2011...
Processing year 2012...
Processing year 2013...
Processing year 2014...
Processing year 2015...
Processing year 2016...
Processing year 2017...
Processing year 2018...
Processing year 2019...
Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...
Processing year 2025...
Processing year 2026...


In [20]:
rain

,prec
time,
2000-01-01 12:00:00,4.962662
2000-01-02 12:00:00,4.615260
2000-01-03 12:00:00,3.587662
2000-01-04 12:00:00,0.001623
2000-01-05 12:00:00,0.389610
...,...
2026-06-11 12:00:00,0.000000
2026-06-12 12:00:00,0.006494
2026-06-13 12:00:00,0.000000


## Get DAILY discharge

In [3]:
hydro = Hydrology()

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code AC9KLRNJ3 to authenticate.


In [7]:
daily_q = hydro.get_discharge(STATION, "vazao")
daily_q

,EstacaoCodigo,NivelConsistencia,Vazao
Data,,,
1966-01-02,66010000,2,104.26
1966-01-03,66010000,2,107.81
1966-01-04,66010000,2,108.53
1966-01-05,66010000,2,107.81
1966-01-06,66010000,2,99.31
...,...,...,...
2025-12-27,66010000,1,125.14
2025-12-28,66010000,1,115.80
2025-12-29,66010000,1,112.90


In [8]:
daily_q.to_parquet(f"/data/CN3S/basins/{BASIN}/Daily_Q_{STATION}.parquet")